In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


# Atenção textual para manchetes financeiras verificadas (Colab GPU)

Execute somente no Google Colab com GPU. O notebook clona ou atualiza o repositório, monta o Drive e reutiliza `MyDrive/latent_fusion_data/article_embedding_cache/`. Usa manchetes verificadas do manifesto comprimido, **nunca corpos de artigos**. A data `decision_date` já incorpora o atraso de um pregão. O estudo compara modelo textual, controle sem conteúdo textual, preços, previsão de excesso zero e carteira de comprar e manter. Os resultados e checkpoints reais serão gravados em uma **nova** pasta, sem alterar a experiência anterior.

O teste de 2022-09 a 2023-11 já foi inspecionado na experiência anterior: esta comparação é diagnóstica, não evidência confirmatória nova. Não pressupomos desempenho superior à previsão nula ou comprar e manter.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute exclusivamente no Google Colab com GPU') from exc
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('PyTorch indisponível; selecione GPU e reinicie')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative GPU: Ambiente de execução → Alterar tipo de ambiente → T4 GPU')
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==2.3.1', 'transformers>=4.33.3,<5'], check=True)
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
PANEL, NEWS = DATA / 'fusion_event_panel.npz', DATA / 'fusion_events.jsonl.gz'
for file in (PANEL, NEWS):
    if not file.is_file():
        raise FileNotFoundError(f'Envie o artefato auditado para {file}')
OUT = DATA / 'text_attention_results'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))
{'gpu': torch.cuda.get_device_name(0), 'painel': str(PANEL), 'notícias': str(NEWS), 'saída': str(OUT)}

## Ajuste e avaliação cronológica

Para cada ticker, no máximo oito manchetes dos últimos cinco pregões. A novidade compara embeddings somente com até dez eventos de datas de disponibilidade anteriores, excluindo eventos do mesmo dia. A consulta baseada em MOMENT, volatilidade passada e retorno conhecido escolhe as manchetes por autoatenção e atenção cruzada; a cabeça só lê o texto atendido. A ablação recebe somente máscaras/contagem e idade das notícias. Ambos os modelos emitem exatamente zero quando não há manchete recente. Parada antecipada usa a validação, sem ajustar ao teste.

As carteiras têm pesos positivos, custos de 10 bps por nocional comprado/vendido e as mesmas datas. O retorno de cinco pregões sobrepõe amostras consecutivas. Ausência de manchete não implica ausência de notícias; a cobertura auditável termina em 30/11/2023, com lacuna em janeiro de 2024. O universo ex post inclui ETFs, sujeito a viés de seleção/sobrevivência. Comparações não demonstram causalidade nem garantem alpha futuro.

In [ ]:
from IPython.display import Image, display
from src.fusion.text_attention_experiment import run_experiment

result = run_experiment(PANEL, NEWS, OUT, cache_dir=DATA / 'article_embedding_cache', cost_bps=10)
display(result['signal'], result['portfolio'], result['split_coverage'])
display(result['histories']['price'], result['histories']['text'], result['histories']['metadata'])
display(result['coverage'].pivot(index='month', columns='ticker', values='events'))
display(Image(filename=str(OUT / 'text_attention_vs_controls.png')))
{'relatório': str(OUT / 'results.md'), 'curvas': str(OUT / 'equity_curves.csv'), 'cache': str(result['cache'])}